# 👑 Gauss-Jordan con Pivoteo Parcial — Solución Maestra Completa
**Equipo 8** — *Cómputo Científico, Estabilidad Numérica y Machine Learning*

Este cuaderno contiene la suite completa de producción desarrollada para la clase de Métodos Numéricos:
1. Gauss-Jordan sin pivoteo (versión frágil)
2. Gauss-Jordan con pivoteo parcial robusto
3. Inversa de una matriz $[A | I] \to [I | A^{-1}]$
4. Experimento obligatorio: Perturbación $\epsilon = 10^{-15}$
5. Matrices de Hilbert: Mal condicionamiento $\kappa(A)$ y residuo vs error
6. Benchmark de complejidad: Resolver directo vs Calcular la inversa
7. Aplicación moderna: Regresión en Machine Learning vía Ecuaciones Normales

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=6, suppress=True)
print("Entorno científico listo.")

## 1. Algoritmo de Gauss-Jordan con Pivoteo Parcial

In [ ]:
def gauss_jordan_con_pivoteo(A, b, verbose=False):
    """
    Resuelve el sistema Ax = b mediante Gauss-Jordan con Pivoteo Parcial.
    """
    A = np.array(A, dtype=np.float64, copy=True)
    b = np.array(b, dtype=np.float64, copy=True).reshape(-1, 1)
    n = len(b)
    
    # 1. Matriz aumentada [A | b]
    Ab = np.hstack([A, b])
    
    if verbose:
        print("\n--- MATRIZ AUMENTADA INICIAL [A | b] ---")
        print(Ab)
        
    for k in range(n):
        # 2. Búsqueda del pivote parcial en columna k
        fila_max = k + np.argmax(np.abs(Ab[k:, k]))
        pivote_val = Ab[fila_max, k]
        
        if np.isclose(pivote_val, 0.0, atol=1e-14):
            raise ValueError(f"Sistema singular en columna {k}.")
            
        # 3. Intercambio de filas si fila_max != k
        if fila_max != k:
            Ab[[k, fila_max]] = Ab[[fila_max, k]]
            if verbose:
                print(f"[PIVOTEO] Swap Fila {k} <---> Fila {fila_max} (Pivote = {pivote_val:.4f})")
                print(Ab)
                
        # 4. Normalizar la fila pivote para tener 1.0 en la diagonal
        Ab[k] = Ab[k] / Ab[k, k]
        
        # 5. Eliminación bidireccional
        for i in range(n):
            if i != k:
                factor = Ab[i, k]
                Ab[i] = Ab[i] - factor * Ab[k]
                
        if verbose:
            print(f"\n--- Columna {k} reducida ---")
            print(Ab)
            
    # 6. Extracción de solución
    x = Ab[:, -1]
    return x

## 2. Gauss-Jordan sin Pivoteo (Demostración de Fragilidad)

In [ ]:
def gauss_jordan_sin_pivoteo(A, b):
    A = np.array(A, dtype=np.float64, copy=True)
    b = np.array(b, dtype=np.float64, copy=True).reshape(-1, 1)
    n = len(b)
    Ab = np.hstack([A, b])
    
    for col in range(n):
        pivote = Ab[col, col]
        if np.isclose(pivote, 0.0, atol=1e-15):
            raise ZeroDivisionError(f"Pivote nulo en fila {col}.")
        Ab[col] = Ab[col] / pivote
        for fila in range(n):
            if fila != col:
                factor = Ab[fila, col]
                Ab[fila] -= factor * Ab[col]
    return Ab[:, -1]

## 3. Función de Análisis: Residuo vs Error Hacia Adelante

In [ ]:
def analizar_solucion(A, b, x_calculado, x_real=None, nombre_metodo="Método"):
    A = np.array(A, dtype=np.float64)
    b = np.array(b, dtype=np.float64).flatten()
    x_hat = np.array(x_calculado, dtype=np.float64).flatten()
    
    r = b - A @ x_hat
    norma_residuo = np.linalg.norm(r, 2)
    
    print(f"\n[{nombre_metodo}]")
    print(f"  -> Solución x_hat       : {x_hat}")
    print(f"  -> Norma Residuo ||r||_2: {norma_residuo:.3e}")
    
    if x_real is not None:
        x_real = np.array(x_real, dtype=np.float64).flatten()
        error_relativo = np.linalg.norm(x_real - x_hat, 2) / np.linalg.norm(x_real, 2)
        print(f"  -> Error hacia adelante : {error_relativo:.3e}")
        return norma_residuo, error_relativo
    return norma_residuo, None

## 🧪 4. Casos de Prueba Iniciales: Regular y Pivote Cero

In [ ]:
# Caso Regular 3x3
A_reg = np.array([[2.0, 1.0, -1.0], [-3.0, -1.0, 2.0], [-2.0, 1.0, 2.0]])
b_reg = np.array([8.0, -11.0, -3.0])
x_pivot = gauss_jordan_con_pivoteo(A_reg, b_reg)
x_numpy = np.linalg.solve(A_reg, b_reg)
analizar_solucion(A_reg, b_reg, x_pivot, x_real=x_numpy, nombre_metodo="Gauss-Jordan con Pivoteo")

# Caso con Pivote Cero
A_swap = np.array([[0.0, 2.0, 1.0], [1.0, -1.0, 1.0], [2.0, 1.0, -1.0]])
b_swap = np.array([5.0, 1.0, 4.0])
x_swap = gauss_jordan_con_pivoteo(A_swap, b_swap, verbose=True)
analizar_solucion(A_swap, b_swap, x_swap, nombre_metodo="Gauss-Jordan con Pivote Cero")

## 🔬 5. Experimento Obligatorio: Inestabilidad con $\epsilon = 10^{-15}$

In [ ]:
eps = 1e-15
A_inest = np.array([[eps, 1.0], [1.0, 1.0]])
b_inest = np.array([1.0 + eps, 2.0])
x_exacto = np.array([1.0, 1.0])

print("Probando sin pivoteo:")
try:
    x_sin = gauss_jordan_sin_pivoteo(A_inest, b_inest)
    analizar_solucion(A_inest, b_inest, x_sin, x_real=x_exacto, nombre_metodo="Sin Pivoteo")
except Exception as e:
    print("Sin pivoteo falló con excepción:", e)

print("\nProbando con pivoteo parcial:")
x_con = gauss_jordan_con_pivoteo(A_inest, b_inest)
analizar_solucion(A_inest, b_inest, x_con, x_real=x_exacto, nombre_metodo="Con Pivoteo")

print("\nProbando con numpy.linalg.solve:")
x_np = np.linalg.solve(A_inest, b_inest)
analizar_solucion(A_inest, b_inest, x_np, x_real=x_exacto, nombre_metodo="NumPy Solve")

## 🧠 6. Matrices de Hilbert: Mal Condicionamiento y $\kappa(A)$

In [ ]:
for n in [4, 6, 8, 10]:
    H = np.array([[1.0 / (i + j + 1) for j in range(n)] for i in range(n)])
    x_real = np.ones(n)
    b = H @ x_real
    cond_H = np.linalg.cond(H)
    print(f"\n--- Hilbert {n}x{n} | kappa(H) = {cond_H:.2e} ---")
    x_calc = gauss_jordan_con_pivoteo(H, b)
    analizar_solucion(H, b, x_calc, x_real=x_real, nombre_metodo=f"Gauss-Jordan H_{n}")

## ⏱️ 7. ¿Por qué NUNCA calcular $A^{-1}$? (Benchmark)

In [ ]:
n = 800
np.random.seed(42)
A = np.random.rand(n, n) + np.eye(n) * 10
b = np.random.rand(n)

t0 = time.time()
x_dir = np.linalg.solve(A, b)
t_dir = time.time() - t0

t0 = time.time()
A_inv = np.linalg.inv(A)
x_inv = A_inv @ b
t_inv = time.time() - t0

print(f"1. Solver directo : {t_dir:.4f} s | Residuo: {np.linalg.norm(b - A @ x_dir):.3e}")
print(f"2. Inversa A^-1 * b: {t_inv:.4f} s | Residuo: {np.linalg.norm(b - A @ x_inv):.3e}")
print(f"-> Invertir tomó {t_inv / t_dir:.2f}x más tiempo (~n^3 vs ~1/3 n^3 flops).")

## 🤖 8. Aplicación en Machine Learning: Regresión vía Ecuaciones Normales

In [ ]:
np.random.seed(101)
m = 50
x_data = np.linspace(-3, 3, m)
y_data = 1.5 * (x_data**2) - 2.0 * x_data + 3.0 + np.random.normal(0, 1.5, size=m)

X = np.column_stack([np.ones(m), x_data, x_data**2])
A_ml = X.T @ X
b_ml = X.T @ y_data

beta = gauss_jordan_con_pivoteo(A_ml, b_ml)
print(f"Parámetros óptimos calculados: beta_0 = {beta[0]:.4f}, beta_1 = {beta[1]:.4f}, beta_2 = {beta[2]:.4f}")

x_curva = np.linspace(-3.2, 3.2, 200)
y_pred = np.column_stack([np.ones(len(x_curva)), x_curva, x_curva**2]) @ beta

plt.figure(figsize=(9, 5))
plt.scatter(x_data, y_data, color='crimson', label='Datos Experimentales')
plt.plot(x_curva, y_pred, color='navy', linewidth=2.5, label='Ajuste Gauss-Jordan ML')
plt.title('Regresión Polinomial vía Ecuaciones Normales (Gauss-Jordan)', fontweight='bold')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()